- log_probs 与 entropys 一样都是 token 级别的
    - `[128, 1024]` (bsz, response_len)
        - log_probs: `[128, 1024]`
        - entropys: `[128, 1024]`

- verl.utils.torch_functional.logprobs_from_logits
    - Compute per-token log-probabilities for the given labels.

```python
def logprobs_from_logits(logits, labels, inplace_backward=True):
    """
    Compute per-token log-probabilities for the given labels.

    Uses a Flash-Attention–based cross-entropy (if available) for efficient backward,
    otherwise falls back to a standard log-softmax+gather approach.

    See: https://github.com/pytorch/pytorch/issues/563#issuecomment-330103591

    Args:
        logits (Tensor): Model outputs of shape (..., vocab_size).
        labels (LongTensor): True class indices of shape matching logits[..., :-1].
        inplace_backward (bool): If True and Flash-Attn is available, perform backward in-place.

    Returns:
        Tensor: Log-probabilities of the target labels, shape logits.shape[:-1].
    """
    if FLAH_ATTN_CROSS_ENTROPY_LOSS_AVAILABLE:
        batch_dim = logits.shape[:-1]
        last_dim = logits.shape[-1]
        logits = logits.reshape(-1, last_dim)
        labels = labels.reshape(-1)
        output = logprobs_from_logits_flash_attn(logits, labels, inplace_backward=inplace_backward)
        output = output.view(*batch_dim)
    elif NPU_CROSS_ENTROPY_LOSS_AVAILABLE:
        output = logprobs_from_logits_torch_npu(logits, labels)
    else:
        output = logprobs_from_logits_v2(logits, labels)
    return output
```

### log probs from logits

- 定义
    - (model) logits: $z$, true label: $y$
    - 标准的交叉熵损失，PyTorch 中 reduction='None' 计算的是真实标签 $y$ 对应的负对数概率：

$$
\text{CrossEntropyLoss}(z,y)=-\log p(y|z)=-\log\left(\frac{\exp(z_y)}{\sum_i\exp(z_i)}\right)
$$

- log p = -crossentropyloss

$$
\log p(y|z)=\log\left(\frac{\exp(z_y)}{\sum_i\exp(z_i)}\right)=-\text{CrossEntropyLoss}(z,y)
$$

- log p_y = z_y - logsumexp(z)
    - softmax: $z=(z_1,z_2,\cdots,z_k)$ => $p=(p_1, p_2, \cdots, p_k)$
        - $p_j=\frac{\exp(z_j)}{\sum_i\exp(z_i)}$
    - log-softmax
        - $\log p_j=\log \frac{\exp(z_j)}{\sum_i\exp(z_i)}=\log \exp(z_j) - \log{\sum_i\exp(z_i)}=z_j-\log{\sum_i\exp(z_i)}$
        - $\log p_j=z_j-\text{logsumexp}(z)$
    - $z_y$ 通过 torch.gather 实现；
    - logsumexp 有专门的数值稳定性计算优化
        - $\text{logsumexp}(z)=z_{max}+\log(\sum_j\exp(z_j-z_{max}))$
            - 最大的 z 对应的 exp 项就变成了 $\exp(0)=1$，避免了 overflow；
    - 内存的角度
        - log softmax：`[bsz, seq_len, vocab_size]`
        - z_y - logsumexp(z):
            - z_y: `[bsz, seq_len]`
            - logsumexp(z): `[bsz, seq_len]`

### entropy from logits

> 虽然希望entropy相对高（有一定多样性）但不希望爆炸高（出现乱码）。

- (trl) `ppo_trainer.py`


```python
# logits.shape: (total_tokens, vocab_size)
def entropy_from_logits(logits: torch.Tensor):
    """Calculate entropy from logits."""
    pd = torch.nn.functional.softmax(logits, dim=-1)
    entropy = torch.logsumexp(logits, dim=-1) - torch.sum(pd * logits, dim=-1)
    return entropy

# return: (total_tokens, ), token 级别的熵
```

$$
H=-\sum_vp(v)\log p(v)
$$

- 在 llm 中，就是 generation 生成序列的每个位置 $\pi_\theta(\cdot |q, o_{\lt t})$ 都对应一个词表维度的概率分布

$$
p(v)=\frac{\exp(\text{logits}_v)}{\sum_{v'}\exp(\text{logits}_{v'})}=\frac{\exp(\text{logits}_v)}{Z}
$$

- 则有

$$
\log p(v)=\text{logits}_v-\log Z
$$

- 进一步

$$
H=-\sum_v p(v)\log p(v)=-\sum_v p(v)(\text{logits}_v-\log Z)=\log Z - \sum_v p(v)\text{logits}_v
$$